In [1]:
import requests
from bs4 import BeautifulSoup

Test 1
- get the article text from a random article of the ecg

In [2]:
# get content of a random article
url = 'https://muse.jhu.edu/document/1266/'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')

In [3]:
# get the article content (body)
article_body = soup.find_all(id="body")

In [4]:
# type of article_body object is a beautiful soup ResultSet
type(article_body)

bs4.element.ResultSet

In [5]:
# only 1 element with an id of "body" exists on the article page
len(article_body)

1

In [6]:
# print out the article_body for future analysis
article_body

[<div id="body"><div class="sec no-title">
 <p>After the closing of the Nohra concentration camp on April 12, 1933, it became ever more urgent to establish a new concentration camp in Thüringen. The reason for this was the increasing political opposition from workers’ organizations.</p>
 <p>At the end of October, the choice was made for a camp in the small sanatorium town of Bad Sulza, about 25 kilometers (15.5 miles) from the state capital Weimar. The site chosen was a former hotel built in 1864, which operated as such until 1914. During World War I, the hotel functioned as a hospital. After that, various small businesses operated from it. Several tenants occupied the front section of the building. To the rear was a courtyard, enclosed by two two-story buildings on the longitudinal side and a two-story building on the lateral axis.</p>
 <div class="figure">
 <div class="thumbnail">
 <a data-lightbox="default" data-title="Click to view image in larger frame. Use arrow keys to navigate 

Test 2
- get the article links for the first 3 people in the index search
- navigate to each article for each person
- print out the article text

In [7]:
# set up base url for article pages and for index search
base_article_url = "https://muse.jhu.edu"
base_name_index_url = 'https://muse.jhu.edu/ushmm/index/names'
# create session to save cookies
session = requests.Session()
session.get(base_name_index_url)

<Response [200]>

In [8]:
# get content of the index search page
index_search_response = requests.get(base_name_index_url)
index_search_soup = BeautifulSoup(index_search_response.text, 'lxml')

In [9]:
# list of all people entries
people_entries = index_search_soup.find_all(class_="entry")

In [10]:
# helper function - get soup of the results of a person search
def get_person_search_results(name):
    # params for AJAX request
    params = {
        "action": "get_links",
        "v": "",
        "search_term": name
    }
    # headers for AJAX request
    headers = {
        "Referer": "https://muse.jhu.edu/ushmm/index/names",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest"
    }
    # get results of the person search
    person_response = requests.get(base_name_index_url, params=params, headers=headers)
    person_response.raise_for_status()
    person_result = BeautifulSoup(person_response.text, 'lxml')
    return person_result

In [11]:
# helper function - get dict of article name (place name) with its url (including base url)
def get_associated_article_links(person_soup):
    h3 = person_soup.find("h3")
    h4 = person_soup.find("h4")
    entries = person_soup.select("ol li a")
    articles = {}
    if h3 and h4 and entries:
        for e in entries:
            articles[e.text] = base_article_url + e["href"] 
    return articles

In [12]:
# helper function - get the content of an article given the full url
def get_article_content(url):
    article_response = requests.get(url)
    article_response.raise_for_status()
    article_soup = BeautifulSoup(article_response.text, 'lxml')
    article_body = article_soup.find_all(id="body")
    return article_body

In [13]:
# test all associated articles for first 3 people in the index search
i = 0
for person in people_entries:
    if i >= 3:
        break
    # save person name - in the text field of the entry
    person_name = person.text
    # use helper functions to get the articles associated with that person and print out results
    person_soup = get_person_search_results(person_name)
    article_links = get_associated_article_links(person_soup)
    print(person_name, article_links)
    # for each article associated, use helper function to get content and print out results
    for place, link in article_links.items():
        article_body = get_article_content(link)
        print(place)
        print(article_body)
        print()
    i += 1

Aaron, Willi {'BAD SULZA': 'https://muse.jhu.edu/document/1266/', 'BAMBERG': 'https://muse.jhu.edu/document/1267/'}
BAD SULZA
[<div id="body"><div class="sec no-title">
<p>After the closing of the Nohra concentration camp on April 12, 1933, it became ever more urgent to establish a new concentration camp in Thüringen. The reason for this was the increasing political opposition from workers’ organizations.</p>
<p>At the end of October, the choice was made for a camp in the small sanatorium town of Bad Sulza, about 25 kilometers (15.5 miles) from the state capital Weimar. The site chosen was a former hotel built in 1864, which operated as such until 1914. During World War I, the hotel functioned as a hospital. After that, various small businesses operated from it. Several tenants occupied the front section of the building. To the rear was a courtyard, enclosed by two two-story buildings on the longitudinal side and a two-story building on the lateral axis.</p>
<div class="figure">
<div c

BAMBERG
[<div id="body"><div class="sec no-title">
<p>With the March 9, 1933, Nazi takeover of Bavaria, the Wilhelmsplatz State Court Prison in Bamberg, Oberfranken, became a “protective custody” camp.<a class="rid-fn-text" href="#FN000001" name="FN000001-text"><sup>1</sup></a> Between March and July 1933, it altogether held more than 140 detainees, of whom at least 42 were released. Wilhelmsplatz was one of at least nine small protective custody camps in northern Bavaria, which included the camps of Bayreuth (St. Georgen), Coburg, Hof an der Saale, and Straubing in Oberfranken, and the camps of Aschaffenburg, Hassenberg bei Neustadt, Hassfurt, Schweinfurt, and Würzburg in Unterfranken (after 1935, Mainfranken). According to press reports, Bamberg detained 62 Bavarian People’s Party (BVP) members; at least 42 Communist, Social Democratic, Reichsbanner, and trade union leaders; as many as 7 Jews; 1 Stahlhelm member; 1 Jehovah’s Witness; 1 person who defied the regime’s dairy pricing sch

Aaron, Yitzhak {'MINSK': 'https://muse.jhu.edu/document/3067/', 'MIORY': 'https://muse.jhu.edu/document/3068/'}
MINSK
[<div id="body"><div class="sec no-title">
<p><em>Pre-1941: Minsk, city and capital, raion center, Minsk oblast’, Belorussian SSR; 1941–1944: center of Gebiet Minsk-Land and capital, Generalkommissariat Weissruthenien; post-1991: capital and center, Minsk voblasts’, Republic of Belarus</em></p>
<p>Between 1900 and 1930, Minsk was a predominantly Jewish city, with Yiddish one of four official languages there. At the beginning of Stalin’s rule, Belorussia was targeted for “Russification.” As a result the Jewish population had declined as a proportion of the total number of people in the city to 70,998 (or 29.71 percent of the total) by 1939.</p>
<p>The German invasion began on June 22, 1941. The main thrust of the German surprise attack delivered by Army Group Center was aimed directly through Minsk towards Moscow. The German encirclement forced the Red Army to abandon th

MIORY
[<div id="body"><div class="sec no-title">
<p><em>Pre-1939: Miory (Yiddish: Mior), town, Brasław powiat, Wilno województwo, Poland; 1939–1941: raion center, Vileika oblast’, Belorussian SSR; 1941–1944: Rayon center, Gebiet Glebokie, Generalkommissariat Weissruthenien; post-1991: Miery, raen center, Vitsebsk voblasts’, Republic of Belarus</em></p>
<p>Miory is located 163 kilometers (101 miles) west of Vitebsk. In 1939, the Jewish population in Miory was around 725.</p>
<p>On July 2, 1941, German forces entered Miory. As soon as they arrived, a group of about 30 local Poles led by a local doctor organized a pogrom in which they demolished Jewish residences and shot the rabbi and his wife. Apparently this was done in retaliation for the imprisonment of the parish priest during the Soviet occupation, for which one Jew among the local residents was allegedly responsible.<a class="rid-fn-text" href="#FN000001" name="FN000001-text"><sup>1</sup></a> A number of Poles also volunteered for

Aaronson, Yehoshua Moshe {'SANNIKI': 'https://muse.jhu.edu/document/2418/', 'SIERADZ': 'https://muse.jhu.edu/document/2419/'}
SANNIKI
[<div id="body"><div class="sec no-title">
<p><em>Pre-1939: Sanniki, village, Warsaw województwo, Poland; 1939–1945: Kreis Gasten (later Walderode), Regierungsbezirk Hohensalza, Reichsgau Wartheland; post-1998: województwo mazowieckie, Poland</em></p>
<p>Sanniki is located 69 kilometers (43 miles) north-northeast of Łódź. In 1921, the Jewish population was 315, out of a total of 1,447 inhabitants of the village.</p>
<p>At the time of the German invasion in September 1939, the Luftwaffe bombed Sanniki, setting many houses on fire. Some local Jews fled towards Warsaw, but other refugees <strong> [End Page 100] </strong>arrived in the village. German forces captured Sanniki during the third week of September. In the first months of the occupation, the Jews were made to perform forced labor, and the German authorities confiscated much of their property. Afte